# W3: baseline tích hợp A/B
Chạy từ kernel sạch. Notebook này chưa có kết quả corpus được nghiệm thu. Không dùng fixture predictions để báo Recall. QA là tập phát triển, còn chờ review chéo. Probe PDF dùng pypdf; kiểm thử quyền Django chạy riêng trong repository.
Đặt OHAYO_ROOT tới source đã giải nén (thư mục ghi được); OHAYO_CORPUS tới thư mục chứa THMLN, KTCTMLN, CNXHKH. Trên Kaggle bật Internet cho lần tải model đầu, GPU nếu dùng DEVICE=cuda. Không đưa corpus/index riêng tư lên dataset công khai.

In [ ]:
import os, sys, json, subprocess
from pathlib import Path
ROOT = Path(os.environ.get('OHAYO_ROOT', '/kaggle/working/ohayo-smartlms')).resolve()
CORPUS = Path(os.environ.get('OHAYO_CORPUS', '/kaggle/input/ohayo-corpus/corpus')).resolve()
DEVICE = os.environ.get('OHAYO_DEVICE', 'cpu')  # đổi cuda khi đã bật GPU
assert (ROOT / 'backend/requirements-w3.txt').is_file(), 'Giải nén source vào OHAYO_ROOT trước'
os.environ.setdefault('HF_HOME', str(ROOT / '.private/huggingface'))
OUT = ROOT / 'data/processed/w3-integrated'
ARTIFACTS = ROOT / '.private/w3'
def run(*args):
    subprocess.run([sys.executable, *map(str, args)], cwd=ROOT, check=True, env={**os.environ, 'PYTHONPATH': str(ROOT / 'backend')})
print({'source': str(ROOT), 'corpus': str(CORPUS), 'device': DEVICE})

In [ ]:
run('-m', 'pip', 'install', '-r', 'backend/requirements-w3.txt')
run('-m', 'unittest', 'discover', '-s', 'backend/apps/ai/tests', '-v')

In [ ]:
# Dùng toàn bộ corpus đã chốt; không giới hạn vài PDF rồi gọi đó là baseline toàn bộ.
# Nếu đã nhận trusted chunk artifacts, đặt ARTIFACTS đúng thư mục có corpus_manifest.json và documents/.
if not (ARTIFACTS / 'corpus_manifest.json').exists():
    assert CORPUS.is_dir(), 'Đặt OHAYO_CORPUS tới corpus đã chốt'
    run('backend/scripts/verify_w3_chunks.py', '--corpus', CORPUS, '--output', ARTIFACTS, '--revision', '5617a9f61b028005a4858fdac845db406aefb181')
COMMON = ['scripts/run_w3_baseline.py', '--corpus-manifest', ARTIFACTS / 'corpus_manifest.json', '--storage-root', ARTIFACTS / 'documents', '--qa', ROOT / 'data/qa/qa_baseline.json', '--config', ROOT / 'backend/config/w3/embedding_config.json', '--output-dir', OUT, '--allow-download', '--device', DEVICE]
run(*COMMON, '--validate-only', '--check-tokenizer')

In [ ]:
# Tải BGE-M3 thật, embed toàn bộ chunk, lưu/load FAISS, query top-10 và chấm tất cả QA.
# Có thể lâu trên CPU. Dừng/lỗi không được coi là baseline đã hoàn thành.
run(*COMMON)
metrics = json.loads((OUT / 'metrics.json').read_text(encoding='utf-8'))
print(json.dumps({key: metrics[key] for key in ['total_qa', 'scored_qa', 'retrieval_error_qa', 'page_recall', 'hit_rate', 'authorization_verified']}, ensure_ascii=False, indent=2))
assert metrics['provenance']['reload_checks'] == 3
assert metrics['retrieval_error_qa'] == 0, 'Phải phân tích và xử lý lỗi trước khi nghiệm thu'

In [ ]:
# Tái chạy từ index đã lưu với cùng config/device/offline flag; không embed lại corpus.
run(*COMMON, '--reuse-index')
print('Output:', sorted(p.name for p in OUT.iterdir()))
print('Bàn giao predictions.json, retrieval_results.csv, metrics.json, preflight.json và index/current.json + toàn bộ generation tương ứng qua nơi lưu phù hợp quyền.')

## Chốt W3 trước hết ngày 16/10
Dương review 50 QA Cường; Cường review phần Dương, xử lý KTCTMLN_QA_001, khóa dataset rồi chạy baseline cuối. Lưu notebook đã Run All từ kernel sạch, môi trường/config/hash, lỗi/hit-miss từng mẫu và vị trí artifact. Nếu Page Recall@5 dưới 0,80, báo đúng kết quả và phân tích lỗi; không sửa gold theo retrieval. Ngày 17 chỉ tổng hợp báo cáo. Luồng này không triển khai chatbot sinh câu trả lời, không tự công bố học liệu hay đánh dấu RAG READY.